# 🇸🇦 Arabic Piper TTS Fine-Tuning — Complete Pipeline

This notebook contains the **entire fine-tuning pipeline** for the Piper Arabic `ar_JO-kareem-medium` model.
Run each section top-to-bottom. If your Colab session disconnects, re-run from **Section 1** (setup is fast) and the training will auto-resume from the last checkpoint saved to Google Drive.

### Sections:
1. ⚙️ Environment Setup & GPU Check
2. 📦 Dataset Download & Preparation
3. 🔊 Baseline Benchmark (Before Training)
4. 🏋️ Fine-Tuning & Checkpointing
5. 📊 Export, Evaluation & Comparison

---
## ⚙️ Section 1: Environment Setup

In [ ]:
# 1.1 Clone Repository from GitHub
# ⚠️ Replace YOUR_USERNAME with your actual GitHub username or org.
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/piper-tts-finetuning.git'  # <-- EDIT THIS
REPO_DIR = '/content/piper-tts-finetuning'

if os.path.exists(REPO_DIR):
    print(f'Repository already cloned at {REPO_DIR}. Pulling latest changes...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# 1.2 GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1.3 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.4 Set up Drive Directories
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Arabic-Piper')
subdirs = ['datasets', 'processed', 'checkpoints', 'tensorboard', 'logs', 'outputs', 'metrics']

for s in subdirs:
    d = DRIVE_ROOT / s
    d.mkdir(parents=True, exist_ok=True)
    print(f"Drive directory verified: {d}")

In [ ]:
# 1.5 Install System & Python Dependencies
import sys; print(f'Python version: {sys.version}')
!apt-get update && apt-get install -y espeak-ng libespeak-ng-dev build-essential cython3
!pip install -r requirements.txt
print('\n✅ Core dependencies installed successfully!')

In [ ]:
# 1.6 Install Piper Training Engine & Build C-Extensions (piper_train)
import os

PIPER_SRC = '/content/piper'
if not os.path.exists(PIPER_SRC):
    !git clone https://github.com/rhasspy/piper.git {PIPER_SRC}

!pip install cython setuptools
!pip install -e {PIPER_SRC}/src/python

# Build C-extension for monotonic alignment
!cd {PIPER_SRC}/src/python/piper_train/vits/monotonic_align && python setup.py build_ext --inplace
print('\n✅ Piper training engine (piper_train) installed & compiled successfully!')

---
## 📦 Section 2: Dataset Download & Preparation

Downloads the **Arabic Professional Voice** dataset from Hugging Face, resamples audio to **22,050 Hz 16-bit PCM WAVs**, and splits into train/validation sets.

In [ ]:
# 2.1 Download Dataset & Base Checkpoint
!python scripts/download_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.2 Prepare Dataset into LJSpeech Format
!python scripts/prepare_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.3 Verify Output Statistics
import pandas as pd
from pathlib import Path

processed_dir = Path('/content/drive/MyDrive/Arabic-Piper/processed/experiment001')
train_path = processed_dir / 'train.csv'
val_path = processed_dir / 'val.csv'

if train_path.exists():
    train_df = pd.read_csv(train_path, sep='|', header=None)
    val_df = pd.read_csv(val_path, sep='|', header=None)
    print(f"✓ Processed dataset loaded.")
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples:   {len(val_df)}")
    print("Sample metadata:", train_df.head(2).values)

---
## 🔊 Section 3: Baseline Benchmark (Before Training)

Runs the baseline `ar_JO-kareem-medium` model on 10 benchmark sentences to establish pronunciation quality and RTF metrics **before** fine-tuning.

In [ ]:
# 3.1 Download Base ONNX Model
!mkdir -p /content/drive/MyDrive/Arabic-Piper/checkpoints/base/
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json

In [ ]:
# 3.2 Run Benchmark Synthesis
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx \
    --model-config /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark

In [ ]:
# 3.3 Display Benchmark Report & Play Audio
import json
from IPython.display import Audio, display, HTML
from pathlib import Path

report_file = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_report.json')
if report_file.exists():
    with open(report_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Baseline Average RTF: {data.get('avg_rtf')}")
    print(f"Total Audio Duration: {data.get('total_audio_duration_sec')}s")
    
    # Play first benchmark sample
    sample_wav = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
    if sample_wav.exists():
        display(HTML('<h4>🔊 Baseline Sample 1:</h4>'))
        display(Audio(str(sample_wav)))

---
## 🏋️ Section 4: Fine-Tuning & Checkpointing

Trains the model using PyTorch Lightning. Checkpoints are auto-saved to Google Drive every 5 epochs. If the session disconnects, re-run from Section 1 — training resumes automatically from the last checkpoint.

In [ ]:
# 4.1 Checkpoint Detection & Setup
import os
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
ckpt_dir.mkdir(parents=True, exist_ok=True)

existing_ckpts = sorted(list(ckpt_dir.glob('*.ckpt')))
if existing_ckpts:
    resume_ckpt = str(existing_ckpts[-1])
    print(f"✅ Resuming training from latest checkpoint: {resume_ckpt}")
else:
    base_ckpt = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/base/epoch=5079-step=1682020.ckpt')
    resume_ckpt = str(base_ckpt) if base_ckpt.exists() else ''
    print(f"🆕 Starting fine-tuning from base checkpoint: {resume_ckpt}")

In [ ]:
# 4.2 Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001/lightning_logs

In [ ]:
# 4.3 Execute Training Run
!python -m piper_train \
    --dataset-dir /content/drive/MyDrive/Arabic-Piper/processed/experiment001 \
    --accelerator gpu \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --max-epochs 50 \
    --checkpoint-epochs 5 \
    --default_root_dir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001 \
    --resume_from_checkpoint "{resume_ckpt}"

---
## 📊 Section 5: Export, Evaluation & Comparison

Exports the fine-tuned checkpoint to **ONNX**, runs benchmark synthesis, and compares **Baseline vs Fine-Tuned** metrics side-by-side.

In [ ]:
# 5.1 Export Best Checkpoint to ONNX
import os
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
best_ckpts = sorted(list(ckpt_dir.glob('*.ckpt')))

if best_ckpts:
    target_ckpt = str(best_ckpts[-1])
    output_onnx = '/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx'
    print(f"Exporting checkpoint: {target_ckpt}")
    !python scripts/export_model.py --checkpoint "{target_ckpt}" --output-onnx "{output_onnx}"
else:
    print("No checkpoint found in checkpoints directory.")

In [ ]:
# 5.2 Benchmark Fine-Tuned Model
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark

In [ ]:
# 5.3 Compare Baseline vs Fine-Tuned Metrics
!python scripts/evaluate.py \
    --baseline-report /content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_report.json \
    --finetuned-report /content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark/benchmark_report.json \
    --output-csv /content/drive/MyDrive/Arabic-Piper/metrics/experiment001_comparison.csv

In [ ]:
# 5.4 Side-by-Side Audio Comparison
from IPython.display import Audio, display, HTML
from pathlib import Path

base_audio = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
ft_audio = Path('/content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark/benchmark_01.wav')

if base_audio.exists() and ft_audio.exists():
    display(HTML('<h3>🔊 Baseline (Before Training):</h3>'))
    display(Audio(str(base_audio)))
    display(HTML('<h3>🎙️ Fine-Tuned (After Training):</h3>'))
    display(Audio(str(ft_audio)))
else:
    print('Audio files not found. Make sure training and benchmarking completed.')

---
## ✅ Done!

Your fine-tuned ONNX model is saved at:
```
/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx
```

Download the `.onnx` and `.onnx.json` files to test locally:
```bash
python scripts/test_local.py --mode finetuned --model path/to/ar_JO_finetuned.onnx --text "السَّلَامُ عَلَيْكُمْ"
```